# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedsamymohamad/flyrank_internship_starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1:** "Content updated in the last 30 days saw a 15% higher visibility retention."
**Methodology Question:** Is this causal, or is there selection bias? Editors usually update high-value pages that are already performing well. Are we comparing similar baselines, or just measuring the difference between maintained vs abandoned content?

**Finding 2:** "High AI referral traffic strongly correlates with reduced organic decline."
**Methodology Question:** Since AI traffic is notoriously sparse, was the sample size large enough across different client sectors? How did the split ensure we didn't memorize a few giant publishers who naturally capture AI clicks?

In [1]:
# No code required for this text section, just leaving empty or a placeholder
print("Methodology questions drafted.")

Methodology questions drafted.


## 2. My model under an honest split (before/after)

We train our Random Forest on a Random Split (memorizing time and clients) vs an Honest Time-Aware Split (past predicts future). Random splits almost always inflate scores by leaking future context or memorizing entity behaviors.

In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

df = pd.read_parquet('../outputs/capstone_features.parquet')
feature_cols = ['clicks_feat', 'impressions_feat', 'avg_pos_feat', 'search_volume', 'competition', 'word_count']
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# 1. Random Split (Before)
X = df[feature_cols]
y = df['is_declining_label']
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(X, y, test_size=0.2, random_state=42)

clf_rand = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
clf_rand.fit(X_train_rand, y_train_rand)
auc_rand = roc_auc_score(y_test_rand, clf_rand.predict_proba(X_test_rand)[:, 1])

# 2. Honest Time-Aware Split (After - already implemented in w05)
train_df = df[df['split'] == 'train']
test_df = df[df['split'] == 'test']
clf_honest = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
clf_honest.fit(train_df[feature_cols], train_df['is_declining_label'])
auc_honest = roc_auc_score(test_df['is_declining_label'], clf_honest.predict_proba(test_df[feature_cols])[:, 1])

comparison = pd.DataFrame({
    'Validation Design': ['Random Split (Leaky)', 'Time-Aware Split (Honest)'],
    'ROC-AUC': [f"{auc_rand:.3f}", f"{auc_honest:.3f}"]
})
display(comparison)

,Validation Design,ROC-AUC
0,Random Split (Leaky),0.991
1,Time-Aware Split (Honest),0.991


## 3. Leakage audit

We verify that no single feature is "suspiciously perfect" (importance > 0.9) which would indicate we accidentally included a proxy for the label (like `trend_pct`).

In [3]:
importances = clf_honest.feature_importances_
leakage_check = pd.DataFrame({'Feature': feature_cols, 'Importance': importances}).sort_values('Importance', ascending=False)

display(leakage_check)

if leakage_check['Importance'].max() > 0.90:
    print("WARNING: Severe leakage detected! A feature is acting as a label proxy.")
else:
    print("Audit passed: Importances are distributed naturally without a dominant leaked feature.")

,Feature,Importance
0,clicks_feat,0.775297
1,impressions_feat,0.162244
2,avg_pos_feat,0.038591
5,word_count,0.018159
4,competition,0.002932
3,search_volume,0.002777


Audit passed: Importances are distributed naturally without a dominant leaked feature.


## 4. Claim rewrite

**Boldest (unsafe) sentence:** "Our model accurately predicts which pages will crash next month, proving that editors can use it to stop algorithmic decline."

**Honest rewrite:** "Our model provides a directional risk score identifying pages historically associated with upcoming traffic drops, serving as a decision-support tool for editors to prioritize monitoring." 

In [4]:
# Claim successfully rewritten
print("Safe language applied.")

Safe language applied.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.